In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "69d240dc",
   "metadata": {},
   "source": [
    "# 03 · Gold (local + pandas) — pronta para negócio e preparada para IA\n",
    "\n",
    "```\n",
    "SILVER ──► GOLD (Parquet no lake, fonte da verdade)\n",
    "              ├── Parte A · tabelas de negócio ──► Postgres (schema gold) ──► Metabase\n",
    "              └── Parte B · tabela de chunks   ──► [próximo módulo: embeddings + banco vetorial]\n",
    "```\n",
    "\n",
    "A Gold responde a perguntas de negócio, tem **dono, grão e contrato**, e só é publicada se passar nos **gates de qualidade**.\n",
    "\n",
    "**O que NÃO faremos aqui:** criar o banco vetorial. Na Parte B deixamos a arquitetura **pronta para recebê-lo**: tabela de chunks, PII mascarada, `content_hash` para reprocessar só o que mudou e o contrato da tabela `pgvector`. O banco em si é construído no próximo módulo.\n",
    "\n",
    "**Antes de começar:** rode os notebooks 01 e 02 (versões pandas), com o MinIO no ar. A parte \"Servir ao Postgres\" só funciona na aula, quando o Postgres estiver no ar."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6c83f3c1",
   "metadata": {},
   "source": [
    "## Passo 0 · Conexão, funções de apoio e mapa de colunas"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "88e76190",
   "metadata": {},
   "outputs": [],
   "source": [
    "import hashlib\n",
    "import io\n",
    "import json\n",
    "import os\n",
    "import re\n",
    "from datetime import datetime, timezone\n",
    "\n",
    "import boto3\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "\n",
    "ENDPOINT = os.environ.get(\"MINIO_ENDPOINT\", \"http://localhost:9000\")\n",
    "USUARIO = os.environ.get(\"MINIO_USER\", \"admin\")\n",
    "SENHA = os.environ.get(\"MINIO_PASSWORD\", \"admin12345\")\n",
    "\n",
    "s3 = boto3.client(\"s3\", endpoint_url=ENDPOINT, aws_access_key_id=USUARIO,\n",
    "                  aws_secret_access_key=SENHA, region_name=\"us-east-1\")\n",
    "\n",
    "\n",
    "def listar(bucket, prefixo=\"\"):\n",
    "    achados = []\n",
    "    for pagina in s3.get_paginator(\"list_objects_v2\").paginate(Bucket=bucket, Prefix=prefixo):\n",
    "        achados += [(o[\"Key\"], o[\"Size\"]) for o in pagina.get(\"Contents\", [])]\n",
    "    return achados\n",
    "\n",
    "\n",
    "def ler_prefixo(bucket, prefixo):\n",
    "    partes = [pd.read_parquet(io.BytesIO(s3.get_object(Bucket=bucket, Key=c)[\"Body\"].read()))\n",
    "              for c, _ in listar(bucket, prefixo)]\n",
    "    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()\n",
    "\n",
    "\n",
    "def gravar_parquet(df, chave):\n",
    "    buf = io.BytesIO()\n",
    "    df.to_parquet(buf, index=False, engine=\"pyarrow\")\n",
    "    s3.put_object(Bucket=\"warehouse\", Key=chave, Body=buf.getvalue())\n",
    "\n",
    "\n",
    "def gravar_gold(df, tabela):\n",
    "    for chave, _ in listar(\"warehouse\", f\"gold/{tabela}/\"):\n",
    "        s3.delete_object(Bucket=\"warehouse\", Key=chave)          # Gold é reconstruída por inteiro\n",
    "    gravar_parquet(df, f\"gold/{tabela}/part-0000.parquet\")\n",
    "    return len(df)\n",
    "\n",
    "\n",
    "def para_int(serie):\n",
    "    return pd.to_numeric(serie, errors=\"coerce\").astype(\"Int64\")\n",
    "\n",
    "\n",
    "# Mapa de colunas: se o nome na SUA Silver for outro, ajuste aqui e só aqui.\n",
    "AV_PRODUTO = \"produto_id\"      # avaliacoes: produto avaliado\n",
    "AV_TEXTO = \"comentario\"        # avaliacoes: texto livre\n",
    "VALIDOS = [\"pago\", \"enviado\", \"entregue\"]          # status que contam como venda\n",
    "CANCELADOS = [\"cancelado\", \"devolvido\"]\n",
    "\n",
    "\n",
    "def exigir(df, colunas, nome):\n",
    "    faltam = [c for c in colunas if c not in df.columns]\n",
    "    if faltam:\n",
    "        raise KeyError(f\"{nome}: faltam as colunas {faltam}. Colunas existentes: {list(df.columns)}. Ajuste o mapa de colunas.\")\n",
    "\n",
    "\n",
    "print(\"conectado em\", ENDPOINT)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6a70cf96",
   "metadata": {},
   "source": [
    "## Parte A · Gold para o negócio\n",
    "\n",
    "### Passo 1 · O contrato vem antes do código\n",
    "Antes de calcular qualquer coisa, definimos **o que cada tabela é**. Sem grão e sem dono, o mesmo número aparece diferente em dois painéis.\n",
    "\n",
    "| Tabela | Grão (1 linha =) | Regra de negócio | Dado pessoal |\n",
    "|---|---|---|---|\n",
    "| `vendas_diarias` | um dia | receita = soma de `valor_item` dos pedidos `pago`, `enviado` e `entregue`; cancelados e devolvidos são contados à parte | não |\n",
    "| `clientes_360` | um cliente | métricas de compra; **nome mascarado**, sem e-mail | sim, minimizado |\n",
    "| `avaliacoes_resumo` | um produto | média das notas e % de notas baixas (≤ 2) | não |\n",
    "\n",
    "O contrato fica **no código**, não só no texto, porque os gates e o dicionário de dados vão usá-lo."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "bec0f858",
   "metadata": {},
   "outputs": [],
   "source": [
    "CONTRATOS = {\n",
    "    \"vendas_diarias\": {\n",
    "        \"grao\": \"um dia\", \"dono\": \"Comercial (exemplo: preencha com o responsável)\",\n",
    "        \"atualizacao\": \"reconstruída a cada execução\", \"pii\": False,\n",
    "        \"descricao\": \"Vendas por dia. Receita considera só pedidos pago/enviado/entregue.\",\n",
    "        \"colunas\": {\n",
    "            \"data\": \"Dia do pedido\", \"pedidos\": \"Pedidos válidos no dia\", \"itens\": \"Itens vendidos no dia\",\n",
    "            \"receita\": \"Soma de valor_item dos pedidos válidos (R$)\", \"ticket_medio\": \"receita / pedidos (R$)\",\n",
    "            \"pedidos_cancelados\": \"Pedidos cancelados ou devolvidos no dia\", \"_gold_ts\": \"Quando a Gold foi gerada\"},\n",
    "    },\n",
    "    \"clientes_360\": {\n",
    "        \"grao\": \"um cliente\", \"dono\": \"CRM (exemplo: preencha com o responsável)\",\n",
    "        \"atualizacao\": \"reconstruída a cada execução\", \"pii\": True,\n",
    "        \"descricao\": \"Visão do cliente. O nome é mascarado e o e-mail não entra na Gold (minimização).\",\n",
    "        \"colunas\": {\n",
    "            \"cliente_id\": \"Identificador do cliente\", \"nome_mascarado\": \"Nome com letras ocultas\",\n",
    "            \"estado\": \"UF do cadastro\", \"num_pedidos\": \"Pedidos válidos do cliente\", \"valor_total\": \"Receita do cliente (R$)\",\n",
    "            \"ticket_medio\": \"valor_total / num_pedidos (R$)\", \"primeira_compra\": \"Data da primeira compra\",\n",
    "            \"ultima_compra\": \"Data da última compra\", \"dias_desde_ultima_compra\": \"Dias entre a última compra e o último dia da base\",\n",
    "            \"segmento\": \"sem_compra, ativo (até 90 dias) ou inativo\", \"_gold_ts\": \"Quando a Gold foi gerada\"},\n",
    "    },\n",
    "    \"avaliacoes_resumo\": {\n",
    "        \"grao\": \"um produto\", \"dono\": \"Produto (exemplo: preencha com o responsável)\",\n",
    "        \"atualizacao\": \"reconstruída a cada execução\", \"pii\": False,\n",
    "        \"descricao\": \"Satisfação por produto.\",\n",
    "        \"colunas\": {\n",
    "            \"produto_id\": \"Identificador do produto\", \"produto_nome\": \"Nome do produto\",\n",
    "            \"qtd_avaliacoes\": \"Número de avaliações\", \"nota_media\": \"Média das notas (1 a 5)\",\n",
    "            \"pct_notas_baixas\": \"% de avaliações com nota 1 ou 2\", \"_gold_ts\": \"Quando a Gold foi gerada\"},\n",
    "    },\n",
    "}\n",
    "pd.DataFrame([{\"tabela\": t, \"grao\": c[\"grao\"], \"dono\": c[\"dono\"], \"pii\": c[\"pii\"]} for t, c in CONTRATOS.items()])"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "082f5568",
   "metadata": {},
   "source": [
    "### Passo 2 · Ler a Silver"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "597d8117",
   "metadata": {},
   "outputs": [],
   "source": [
    "cli = ler_prefixo(\"warehouse\", \"silver/clientes/\")\n",
    "prod = ler_prefixo(\"warehouse\", \"silver/produtos/\")\n",
    "ped = ler_prefixo(\"warehouse\", \"silver/pedidos/\")\n",
    "itens = ler_prefixo(\"warehouse\", \"silver/itens_pedido/\")\n",
    "av = ler_prefixo(\"warehouse\", \"silver/avaliacoes/\")\n",
    "\n",
    "exigir(ped, [\"pedido_id\", \"cliente_id\", \"data_pedido\", \"status\"], \"silver.pedidos\")\n",
    "exigir(itens, [\"pedido_id\", \"valor_item\", \"status\", \"data_pedido\"], \"silver.itens_pedido\")\n",
    "exigir(av, [\"avaliacao_id\", \"nota\", AV_PRODUTO, AV_TEXTO], \"silver.avaliacoes\")\n",
    "exigir(cli, [\"cliente_id\", \"nome\", \"estado\"], \"silver.clientes\")\n",
    "\n",
    "for nome, df in ((\"clientes\", cli), (\"produtos\", prod), (\"pedidos\", ped), (\"itens_pedido\", itens), (\"avaliacoes\", av)):\n",
    "    print(f\"{nome:<13}{len(df):>8,} linhas\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "25f7812e",
   "metadata": {},
   "source": [
    "### Passo 3 · Construir as tabelas\n",
    "Duas decisões de governança já aparecem no código:\n",
    "- **Nome mascarado:** a Gold é lida por muita gente (Metabase). Quem só precisa de \"quanto o cliente comprou\" não precisa do nome inteiro.\n",
    "- **E-mail fica de fora:** o que não é necessário para a pergunta de negócio não entra na Gold."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "e2f5ba0b",
   "metadata": {},
   "outputs": [],
   "source": [
    "def mascarar_nome(nome):\n",
    "    \"\"\"'Ana Souza' -> 'A** S****' (mantém a inicial de cada palavra).\"\"\"\n",
    "    if not isinstance(nome, str):\n",
    "        return None\n",
    "    return \" \".join(p[0] + \"*\" * (len(p) - 1) for p in nome.split())\n",
    "\n",
    "\n",
    "AGORA = datetime.now(timezone.utc)\n",
    "\n",
    "itens = itens.copy()\n",
    "itens[\"data\"] = pd.to_datetime(itens[\"data_pedido\"]).dt.date\n",
    "validos = itens[itens[\"status\"].isin(VALIDOS)]\n",
    "\n",
    "# vendas_diarias\n",
    "vendas = (validos.groupby(\"data\")\n",
    "          .agg(pedidos=(\"pedido_id\", \"nunique\"), itens=(\"pedido_id\", \"size\"), receita=(\"valor_item\", \"sum\"))\n",
    "          .reset_index())\n",
    "vendas[\"receita\"] = vendas[\"receita\"].round(2)\n",
    "vendas[\"ticket_medio\"] = (vendas[\"receita\"] / vendas[\"pedidos\"]).round(2)\n",
    "canc = ped[ped[\"status\"].isin(CANCELADOS)].assign(data=lambda d: pd.to_datetime(d[\"data_pedido\"]).dt.date)\n",
    "canc = canc.groupby(\"data\").size().rename(\"pedidos_cancelados\").reset_index()\n",
    "vendas = vendas.merge(canc, on=\"data\", how=\"outer\").fillna(0)\n",
    "for c in (\"pedidos\", \"itens\", \"pedidos_cancelados\"):\n",
    "    vendas[c] = vendas[c].astype(int)\n",
    "vendas = vendas.sort_values(\"data\").reset_index(drop=True)\n",
    "\n",
    "# clientes_360\n",
    "por_cli = (validos.groupby(\"cliente_id\")\n",
    "           .agg(num_pedidos=(\"pedido_id\", \"nunique\"), valor_total=(\"valor_item\", \"sum\"),\n",
    "                primeira_compra=(\"data\", \"min\"), ultima_compra=(\"data\", \"max\"))\n",
    "           .reset_index())\n",
    "por_cli[\"cliente_id\"] = para_int(por_cli[\"cliente_id\"])\n",
    "c360 = cli[[\"cliente_id\", \"nome\", \"estado\"]].merge(por_cli, on=\"cliente_id\", how=\"left\")\n",
    "c360[\"nome_mascarado\"] = c360[\"nome\"].map(mascarar_nome)\n",
    "c360 = c360.drop(columns=\"nome\")                                   # o nome completo não sai da Silver\n",
    "c360[\"num_pedidos\"] = c360[\"num_pedidos\"].fillna(0).astype(int)\n",
    "c360[\"valor_total\"] = c360[\"valor_total\"].fillna(0).round(2)\n",
    "c360[\"ticket_medio\"] = (c360[\"valor_total\"] / c360[\"num_pedidos\"].where(c360[\"num_pedidos\"] > 0)).round(2)\n",
    "ref = pd.to_datetime(itens[\"data\"]).max()                          # \"hoje\" = último dia da base (resultado reprodutível)\n",
    "c360[\"dias_desde_ultima_compra\"] = (ref - pd.to_datetime(c360[\"ultima_compra\"])).dt.days\n",
    "c360[\"segmento\"] = np.select([c360[\"num_pedidos\"] == 0, c360[\"dias_desde_ultima_compra\"] <= 90],\n",
    "                             [\"sem_compra\", \"ativo\"], default=\"inativo\")\n",
    "c360 = c360[[\"cliente_id\", \"nome_mascarado\", \"estado\", \"num_pedidos\", \"valor_total\", \"ticket_medio\",\n",
    "             \"primeira_compra\", \"ultima_compra\", \"dias_desde_ultima_compra\", \"segmento\"]]\n",
    "\n",
    "# avaliacoes_resumo\n",
    "av = av.copy()\n",
    "av[AV_PRODUTO] = para_int(av[AV_PRODUTO])\n",
    "resumo_av = (av.groupby(AV_PRODUTO)\n",
    "             .agg(qtd_avaliacoes=(\"avaliacao_id\", \"size\"), nota_media=(\"nota\", \"mean\"),\n",
    "                  pct_notas_baixas=(\"nota\", lambda s: 100 * (s <= 2).mean()))\n",
    "             .reset_index().rename(columns={AV_PRODUTO: \"produto_id\"}))\n",
    "resumo_av[\"nota_media\"] = resumo_av[\"nota_media\"].round(2)\n",
    "resumo_av[\"pct_notas_baixas\"] = resumo_av[\"pct_notas_baixas\"].round(1)\n",
    "nomes = prod[[\"produto_id\", \"nome\"]].rename(columns={\"nome\": \"produto_nome\"})\n",
    "resumo_av = resumo_av.merge(nomes, on=\"produto_id\", how=\"left\")[\n",
    "    [\"produto_id\", \"produto_nome\", \"qtd_avaliacoes\", \"nota_media\", \"pct_notas_baixas\"]]\n",
    "\n",
    "gold = {\"vendas_diarias\": vendas, \"clientes_360\": c360, \"avaliacoes_resumo\": resumo_av}\n",
    "for t, df in gold.items():\n",
    "    df[\"_gold_ts\"] = AGORA\n",
    "    print(f\"{t:<18}{len(df):>7,} linhas\")\n",
    "c360.head(5)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "04bca579",
   "metadata": {},
   "source": [
    "### Passo 4 · Gates: a Gold só é publicada se passar\n",
    "Um gate é uma verificação que **bloqueia** a publicação. Três tipos:\n",
    "1. **Grão e chaves:** a tabela respeita o contrato?\n",
    "2. **Reconciliação:** o total da Gold bate com a Silver, calculado de outro jeito?\n",
    "3. **Governança:** toda coluna tem descrição? Algum dado pessoal que não devia estar aqui?"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "02669dcf",
   "metadata": {},
   "outputs": [],
   "source": [
    "PII_PROIBIDA = {\"nome\", \"email\", \"cpf\", \"telefone\"}      # colunas que nunca podem aparecer na Gold\n",
    "\n",
    "\n",
    "def checar(tabela, df):\n",
    "    c = CONTRATOS[tabela]\n",
    "    regras = [(\"todas as colunas têm descrição no contrato\", set(df.columns) <= set(c[\"colunas\"])),\n",
    "              (\"nenhuma coluna de PII bruta\", not (PII_PROIBIDA & set(df.columns)))]\n",
    "    if tabela == \"vendas_diarias\":\n",
    "        regras += [(\"grão: 1 linha por data\", not df[\"data\"].duplicated().any()),\n",
    "                   (\"data sem nulos\", df[\"data\"].notna().all()), (\"receita >= 0\", (df[\"receita\"] >= 0).all())]\n",
    "    if tabela == \"clientes_360\":\n",
    "        regras += [(\"grão: 1 linha por cliente\", not df[\"cliente_id\"].duplicated().any()),\n",
    "                   (\"cliente_id sem nulos\", df[\"cliente_id\"].notna().all()),\n",
    "                   (\"todos os clientes da Silver estão aqui\", len(df) == len(cli))]\n",
    "    if tabela == \"avaliacoes_resumo\":\n",
    "        regras += [(\"grão: 1 linha por produto\", not df[\"produto_id\"].duplicated().any()),\n",
    "                   (\"nota_media entre 1 e 5\", df[\"nota_media\"].between(1, 5).all())]\n",
    "    return pd.DataFrame([{\"tabela\": tabela, \"regra\": r, \"status\": \"OK\" if bool(ok) else \"ERRO\"} for r, ok in regras])\n",
    "\n",
    "\n",
    "def reconciliar():\n",
    "    \"\"\"Recalcula a receita por OUTRO caminho (pelos pedidos válidos) e compara com a Gold.\"\"\"\n",
    "    ids_validos = set(ped.loc[ped[\"status\"].isin(VALIDOS), \"pedido_id\"])\n",
    "    esperado = round(float(itens.loc[itens[\"pedido_id\"].isin(ids_validos), \"valor_item\"].sum()), 2)\n",
    "    return pd.DataFrame([\n",
    "        {\"tabela\": \"vendas_diarias\", \"regra\": f\"receita Gold = receita Silver ({esperado:,.2f})\",\n",
    "         \"status\": \"OK\" if abs(round(float(vendas[\"receita\"].sum()), 2) - esperado) < 0.01 else \"ERRO\"},\n",
    "        {\"tabela\": \"vendas_diarias\", \"regra\": f\"pedidos Gold = pedidos válidos Silver ({len(ids_validos)})\",\n",
    "         \"status\": \"OK\" if int(vendas[\"pedidos\"].sum()) == len(ids_validos) else \"ERRO\"}])\n",
    "\n",
    "\n",
    "def rodar_gates(tabelas):\n",
    "    rel = pd.concat([checar(t, df) for t, df in tabelas.items()] + [reconciliar()], ignore_index=True)\n",
    "    return rel\n",
    "\n",
    "\n",
    "gates = rodar_gates(gold)\n",
    "print(gates[\"status\"].value_counts().to_string())\n",
    "gates"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "32b425fb",
   "metadata": {},
   "source": [
    "**Demonstração: o gate funcionando.** Vamos estragar uma cópia (uma linha repetida em `vendas_diarias`) e ver o que acontece."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "d3055a2f",
   "metadata": {},
   "outputs": [],
   "source": [
    "ruim = pd.concat([vendas, vendas.head(1)], ignore_index=True)\n",
    "r = checar(\"vendas_diarias\", ruim)\n",
    "print(r[r[\"status\"] == \"ERRO\"].to_string(index=False))\n",
    "print(\"\\n-> publicação BLOQUEADA\" if (r[\"status\"] == \"ERRO\").any() else \"-> publicação liberada\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1b932c0e",
   "metadata": {},
   "source": [
    "### Passo 5 · Publicar no lake\n",
    "A Gold é gravada no storage (**fonte da verdade**). O contrato vai junto, como um objeto em `gold/_contratos/`, para que quem consome saiba o que está lendo."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "98843301",
   "metadata": {},
   "outputs": [],
   "source": [
    "if (gates[\"status\"] == \"ERRO\").any():\n",
    "    raise RuntimeError(\"Gates falharam: a Gold NÃO foi publicada.\\n\" + gates[gates[\"status\"] == \"ERRO\"].to_string(index=False))\n",
    "\n",
    "for t, df in gold.items():\n",
    "    print(f\"{t:<18}{gravar_gold(df, t):>7,} linhas -> s3://warehouse/gold/{t}/\")\n",
    "\n",
    "s3.put_object(Bucket=\"warehouse\", Key=\"gold/_contratos/contratos.json\",\n",
    "              Body=json.dumps(CONTRATOS, ensure_ascii=False, indent=2).encode(\"utf-8\"))\n",
    "print(\"\\ncontrato publicado em s3://warehouse/gold/_contratos/contratos.json\")\n",
    "for chave, tam in listar(\"warehouse\", \"gold/\"):\n",
    "    print(f\"{tam:>9,}  {chave}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "97398833",
   "metadata": {},
   "source": [
    "### Passo 6 · Servir ao Postgres e ao Metabase *(na aula)*\n",
    "O Metabase não lê Parquet: ele fala com um banco SQL. Por isso **copiamos** a Gold para o Postgres (schema `gold`).\n",
    "O lake continua sendo a fonte da verdade; o Postgres é uma **cópia de serviço** que pode ser apagada e refeita.\n",
    "\n",
    "Duas lições de governança:\n",
    "- O Metabase deve usar um usuário **somente leitura**. Quem faz BI não precisa (nem deve) poder alterar a Gold.\n",
    "- `if_exists=\"replace\"` apaga a tabela e **perde as permissões**. Por isso concedemos `SELECT` por padrão no schema.\n",
    "\n",
    "*Se o Postgres ainda não estiver no ar, esta célula avisa e você segue para a Parte B.*"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "0514c80e",
   "metadata": {},
   "outputs": [],
   "source": [
    "from sqlalchemy import create_engine, text\n",
    "\n",
    "PG_HOST = os.environ.get(\"POSTGRES_HOST\", \"localhost\")\n",
    "PG_PORT = os.environ.get(\"POSTGRES_PORT\", \"5432\")\n",
    "PG_DB = os.environ.get(\"POSTGRES_DB\", \"lake\")                 # ajuste ao docker-compose da aula\n",
    "PG_USER = os.environ.get(\"POSTGRES_USER\", \"aula\")\n",
    "PG_SENHA = os.environ.get(\"POSTGRES_PASSWORD\", \"aula123\")     # só laboratório; em produção, cofre de segredos\n",
    "RO_USER = \"metabase_ro\"\n",
    "RO_SENHA = os.environ.get(\"METABASE_PG_PASSWORD\", \"leitor123\")\n",
    "\n",
    "\n",
    "def publicar_postgres(tabelas, schema=\"gold\"):\n",
    "    eng = create_engine(f\"postgresql+psycopg2://{PG_USER}:{PG_SENHA}@{PG_HOST}:{PG_PORT}/{PG_DB}\")\n",
    "    with eng.begin() as con:\n",
    "        con.execute(text(f\"CREATE SCHEMA IF NOT EXISTS {schema}\"))\n",
    "    for nome, df in tabelas.items():\n",
    "        df.to_sql(nome, eng, schema=schema, if_exists=\"replace\", index=False, method=\"multi\", chunksize=1000)\n",
    "    with eng.begin() as con:\n",
    "        con.execute(text(f\"\"\"\n",
    "            DO $$ BEGIN\n",
    "              IF NOT EXISTS (SELECT FROM pg_roles WHERE rolname = '{RO_USER}') THEN\n",
    "                CREATE ROLE {RO_USER} LOGIN PASSWORD '{RO_SENHA}';\n",
    "              END IF;\n",
    "            END $$;\"\"\"))\n",
    "        con.execute(text(f\"GRANT USAGE ON SCHEMA {schema} TO {RO_USER}\"))\n",
    "        con.execute(text(f\"GRANT SELECT ON ALL TABLES IN SCHEMA {schema} TO {RO_USER}\"))\n",
    "        con.execute(text(f\"ALTER DEFAULT PRIVILEGES IN SCHEMA {schema} GRANT SELECT ON TABLES TO {RO_USER}\"))\n",
    "    return eng\n",
    "\n",
    "\n",
    "try:\n",
    "    eng = publicar_postgres(gold)\n",
    "    with eng.connect() as con:\n",
    "        for t in gold:\n",
    "            n = con.execute(text(f\"SELECT count(*) FROM gold.{t}\")).scalar()\n",
    "            print(f\"gold.{t:<18}{n:>7,} linhas no Postgres\")\n",
    "except Exception as e:\n",
    "    print(\"Postgres ainda não está disponível — pule esta célula por enquanto.\")\n",
    "    print(\"Motivo:\", str(e).splitlines()[0][:160])"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "3b13aa80",
   "metadata": {},
   "source": [
    "**Conectar o Metabase** (<http://localhost:3000>): *Admin → Databases → Add database → PostgreSQL*\n",
    "- **Host:** `postgres` (nome do serviço no Docker; use `localhost` se o Metabase não estiver em container)\n",
    "- **Port:** `5432` · **Database:** o do compose · **Username:** `metabase_ro` · **Password:** a de `METABASE_PG_PASSWORD`\n",
    "- Depois de salvar, use *Sync database schema now*. Crie uma pergunta: receita por dia a partir de `gold.vendas_diarias`.\n",
    "\n",
    "**Teste de governança:** tente criar uma tabela ou apagar uma linha usando o usuário `metabase_ro`. Deve falhar. Por quê é bom que falhe?"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "8e364ea2",
   "metadata": {},
   "source": [
    "## Parte B · Gold preparada para IA\n",
    "\n",
    "Um banco vetorial guarda **trechos de texto (chunks)** e seus **embeddings**, para busca por significado. O banco só é criado no próximo módulo, mas as decisões que o tornam seguro e barato **precisam ser tomadas agora**, na Gold:\n",
    "\n",
    "```\n",
    " Silver ──► [mascarar PII] ──► [montar chunks] ──► [content_hash] ──► gold/chunks (Parquet no lake)\n",
    "                                                                              │\n",
    "                                              próximo módulo ─────────────────┘\n",
    "                                              embeddings ──► banco vetorial (pgvector)\n",
    "```\n",
    "\n",
    "| Decisão | Por que agora |\n",
    "|---|---|\n",
    "| **PII mascarada antes** | um embedding não se \"desfaz\": se o telefone entrou no vetor, ele está lá. Depois de gerado, o jeito é refazer. |\n",
    "| **`content_hash`** | gerar embedding custa dinheiro. Com o hash, reprocessamos **só o que mudou**. |\n",
    "| **Lake como fonte da verdade** | o banco vetorial é uma cópia derivada: dá para recriá-lo a partir dos chunks. |\n",
    "| **Contrato da tabela `pgvector`** | o próximo módulo recebe um formato combinado, não uma surpresa. |"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "2a3950cd",
   "metadata": {},
   "source": [
    "### Passo 7 · Classificar o dado pessoal e definir a política\n",
    "Antes de mascarar, decidimos **o que é dado pessoal** e **o que fazer com ele**.\n",
    "\n",
    "| Campo | Classe | Política |\n",
    "|---|---|---|\n",
    "| `nome` | identificador direto | mascarado na Gold; **nunca** entra em chunk |\n",
    "| `email`, CPF, telefone | identificador direto | fora da Gold; se aparecer em texto livre, vira `[EMAIL]`, `[CPF]`, `[TELEFONE]` |\n",
    "| `cliente_id` | pseudônimo (permite ligar à pessoa) | **fora** dos metadados do chunk |\n",
    "| `comentario` (texto livre) | pode conter qualquer coisa | varrido e mascarado |\n",
    "| `estado`, `produto_id`, `nota` | não identificam sozinhos | podem ir como metadado |\n",
    "\n",
    "**Limite honesto:** expressões regulares pegam e-mail, CPF e telefone, mas **não pegam nomes próprios** escritos no meio de uma frase. Isso exige reconhecimento de entidades (NER), que fica para o módulo de IA. Por isso existe o gate de varredura no Passo 10."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "ae45bb63",
   "metadata": {},
   "outputs": [],
   "source": [
    "PADROES_PII = [\n",
    "    (\"EMAIL\", re.compile(r\"[\\w.+-]+@[\\w-]+(?:\\.[\\w-]+)+\")),\n",
    "    (\"CPF\", re.compile(r\"(?<!\\d)\\d{3}\\.?\\d{3}\\.?\\d{3}-?\\d{2}(?!\\d)\")),          # antes do telefone, que é mais genérico\n",
    "    (\"TELEFONE\", re.compile(r\"(?<!\\d)(?:\\+?55\\s?)?(?:\\(?\\d{2}\\)?\\s?)?9?\\d{4}[-\\s]?\\d{4}(?!\\d)\")),\n",
    "]\n",
    "VERSAO_POLITICA_PII = \"v1\"\n",
    "\n",
    "\n",
    "def mascarar_pii(texto):\n",
    "    if not isinstance(texto, str):\n",
    "        return \"\"\n",
    "    for rotulo, padrao in PADROES_PII:\n",
    "        texto = padrao.sub(f\"[{rotulo}]\", texto)\n",
    "    return texto\n",
    "\n",
    "\n",
    "def tem_pii(texto):\n",
    "    return any(p.search(texto) for _, p in PADROES_PII)\n",
    "\n",
    "\n",
    "exemplos = [\"Ótimo produto! Me ligue em (81) 98888-7777 ou escreva para maria.silva@exemplo.com.\",\n",
    "            \"Meu CPF 123.456.789-09 está no cadastro, chegou rápido.\",\n",
    "            \"Chegou em 3 dias, nota 10.\"]\n",
    "for t in exemplos:\n",
    "    print(\"antes :\", t)\n",
    "    print(\"depois:\", mascarar_pii(t), \"\\n\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a78e68dc",
   "metadata": {},
   "source": [
    "### Passo 8 · Montar os chunks\n",
    "Aqui, **1 avaliação = 1 chunk**: o texto é curto e cabe inteiro. Textos longos (manuais, contratos) precisam ser **quebrados** em pedaços com sobreposição; essa estratégia é decidida no módulo vetorial.\n",
    "\n",
    "Cada chunk leva o **contexto** junto (produto e nota), porque um comentário como \"chegou quebrado\" só faz sentido sabendo de qual produto se fala."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "317e6b02",
   "metadata": {},
   "outputs": [],
   "source": [
    "def hash_conteudo(texto):\n",
    "    normalizado = \" \".join(texto.lower().split())            # ignora maiúsculas e espaços repetidos\n",
    "    return hashlib.sha256(normalizado.encode(\"utf-8\")).hexdigest()\n",
    "\n",
    "\n",
    "def montar_chunks(av, prod):\n",
    "    nomes = prod[[\"produto_id\", \"nome\"]].rename(columns={\"nome\": \"produto_nome\"})\n",
    "    df = av.assign(produto_id=para_int(av[AV_PRODUTO])).merge(nomes, on=\"produto_id\", how=\"left\")\n",
    "    df[\"comentario_limpo\"] = df[AV_TEXTO].map(mascarar_pii).str.strip()\n",
    "    df = df[df[\"comentario_limpo\"] != \"\"].copy()\n",
    "    df[\"texto\"] = (\"Produto: \" + df[\"produto_nome\"].fillna(\"desconhecido\") + \" | Nota: \" + df[\"nota\"].astype(str)\n",
    "                   + \"/5 | Avaliação: \" + df[\"comentario_limpo\"])\n",
    "    # minimização: nenhum cliente_id, nome ou e-mail nos metadados\n",
    "    df[\"metadata\"] = [json.dumps({\"produto_id\": None if pd.isna(p) else int(p), \"nota\": int(n), \"origem\": \"silver.avaliacoes\"})\n",
    "                      for p, n in zip(df[\"produto_id\"], df[\"nota\"])]\n",
    "    out = pd.DataFrame({\n",
    "        \"chunk_id\": \"avaliacao:\" + df[\"avaliacao_id\"].astype(str),\n",
    "        \"fonte\": \"silver.avaliacoes\", \"fonte_id\": df[\"avaliacao_id\"].astype(str),\n",
    "        \"texto\": df[\"texto\"].values, \"metadata\": df[\"metadata\"].values,\n",
    "        \"content_hash\": [hash_conteudo(t) for t in df[\"texto\"]],\n",
    "        \"politica_pii\": VERSAO_POLITICA_PII})\n",
    "    return out.reset_index(drop=True)\n",
    "\n",
    "\n",
    "novos = montar_chunks(av, prod)\n",
    "print(len(novos), \"chunks de\", len(av), \"avaliações (sem comentário não vira chunk)\")\n",
    "novos[[\"chunk_id\", \"texto\", \"content_hash\"]].head(4)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "e0d92076",
   "metadata": {},
   "source": [
    "### Passo 9 · `content_hash`: reprocessar só o que mudou\n",
    "O próximo módulo vai gerar um embedding por chunk, e cada embedding custa tempo e dinheiro. Se amanhã só 3 comentários mudarem, não faz sentido refazer os outros 10.000.\n",
    "\n",
    "A regra: comparar o `content_hash` de hoje com o da última execução.\n",
    "\n",
    "| Situação | Ação no próximo módulo |\n",
    "|---|---|\n",
    "| chunk novo | gerar embedding |\n",
    "| hash diferente | gerar de novo (o texto mudou) |\n",
    "| hash igual | **manter** o embedding que já existe |\n",
    "| chunk sumiu | remover do banco vetorial |"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "8dfc2c8c",
   "metadata": {},
   "outputs": [],
   "source": [
    "COLS_ESTADO = [\"chunk_id\", \"content_hash\", \"embedding_status\", \"modelo_embedding\", \"embedding_em\"]\n",
    "\n",
    "\n",
    "def sincronizar(novos, anteriores):\n",
    "    \"\"\"Compara com a execução anterior. Só 'novo' e 'alterado' ficam pendentes de embedding.\"\"\"\n",
    "    out = novos.copy()\n",
    "    if anteriores.empty:\n",
    "        out[\"acao\"] = \"novo\"\n",
    "        out[\"embedding_status\"], out[\"modelo_embedding\"], out[\"embedding_em\"] = \"pendente\", None, pd.NaT\n",
    "        removidos = anteriores\n",
    "    else:\n",
    "        ant = anteriores[COLS_ESTADO].rename(columns={c: c + \"_ant\" for c in COLS_ESTADO if c != \"chunk_id\"})\n",
    "        out = out.merge(ant, on=\"chunk_id\", how=\"left\")\n",
    "        out[\"acao\"] = np.select([out[\"content_hash_ant\"].isna(), out[\"content_hash\"] != out[\"content_hash_ant\"]],\n",
    "                                [\"novo\", \"alterado\"], default=\"inalterado\")\n",
    "        mantem = out[\"acao\"] == \"inalterado\"\n",
    "        out[\"embedding_status\"] = np.where(mantem, out[\"embedding_status_ant\"], \"pendente\")\n",
    "        out[\"modelo_embedding\"] = np.where(mantem, out[\"modelo_embedding_ant\"], None)\n",
    "        out[\"embedding_em\"] = out[\"embedding_em_ant\"].where(mantem, pd.NaT)\n",
    "        out = out.drop(columns=[c for c in out.columns if c.endswith(\"_ant\")])\n",
    "        removidos = anteriores[~anteriores[\"chunk_id\"].isin(out[\"chunk_id\"])]\n",
    "    out[\"atualizado_em\"] = datetime.now(timezone.utc)\n",
    "    return out, removidos\n",
    "\n",
    "\n",
    "# simulação (NÃO grava nada): imagine que o próximo módulo já gerou todos os embeddings...\n",
    "anteriores = novos.assign(embedding_status=\"embedado\", modelo_embedding=\"modelo-exemplo\", embedding_em=pd.Timestamp.now(tz=\"UTC\"))\n",
    "\n",
    "# ...e que, no dia seguinte, um comentário mudou, um sumiu e um chegou\n",
    "id_removido, id_alterado = novos[\"fonte_id\"].iloc[0], novos[\"fonte_id\"].iloc[1]\n",
    "av_amanha = av[av[\"avaliacao_id\"].astype(str) != id_removido].copy()\n",
    "av_amanha.loc[av_amanha[\"avaliacao_id\"].astype(str) == id_alterado, AV_TEXTO] = \"Mudei de ideia: produto ruim, chegou com defeito.\"\n",
    "av_amanha = pd.concat([av_amanha, av.head(1).assign(avaliacao_id=900001, **{AV_TEXTO: \"Comentário novo de amanhã.\"})],\n",
    "                      ignore_index=True)\n",
    "\n",
    "amanha, removidos = sincronizar(montar_chunks(av_amanha, prod), anteriores)\n",
    "print(amanha[\"acao\"].value_counts().to_string())\n",
    "print(\"removidos:\", len(removidos))\n",
    "print(f\"-> embeddings a gerar: {(amanha['acao'] != 'inalterado').sum()} de {len(amanha)}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "de38f5f8",
   "metadata": {},
   "source": [
    "### Passo 10 · Gates de prontidão e publicação dos chunks\n",
    "Os chunks só seguem para o próximo módulo se passarem em verificações próprias. O gate de varredura procura PII **de novo**, sem confiar no mascaramento anterior."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "30f9738d",
   "metadata": {},
   "outputs": [],
   "source": [
    "def gates_chunks(ch):\n",
    "    chaves_proibidas = {\"cliente_id\", \"nome\", \"email\", \"cpf\", \"telefone\"}\n",
    "    regras = [\n",
    "        (\"chunk_id único\", not ch[\"chunk_id\"].duplicated().any()),\n",
    "        (\"texto nunca vazio\", ch[\"texto\"].str.strip().ne(\"\").all()),\n",
    "        (\"sem PII residual no texto (nova varredura)\", not ch[\"texto\"].map(tem_pii).any()),\n",
    "        (\"tamanho <= 1000 caracteres\", (ch[\"texto\"].str.len() <= 1000).all()),\n",
    "        (\"content_hash confere com o texto\", (ch[\"texto\"].map(hash_conteudo) == ch[\"content_hash\"]).all()),\n",
    "        (\"metadados sem cliente_id/nome/e-mail\", not any(chaves_proibidas & set(json.loads(m)) for m in ch[\"metadata\"])),\n",
    "    ]\n",
    "    return pd.DataFrame([{\"regra\": r, \"status\": \"OK\" if bool(ok) else \"ERRO\"} for r, ok in regras])\n",
    "\n",
    "\n",
    "# anterior = o que já foi publicado (se houver), para herdar o estado dos embeddings\n",
    "anterior_pub = ler_prefixo(\"warehouse\", \"gold/chunks/\")\n",
    "chunks, removidos = sincronizar(novos, anterior_pub)\n",
    "g = gates_chunks(chunks)\n",
    "print(g.to_string(index=False))\n",
    "\n",
    "# prova de que o gate funciona: um texto com telefone escapando do mascaramento\n",
    "vazado = chunks.head(1).assign(texto=\"Produto: X | Nota: 5/5 | Avaliação: ligue 81 98888-7777\")\n",
    "print(\"\\nteste do gate com vazamento:\", gates_chunks(vazado).query(\"status == 'ERRO'\")[\"regra\"].tolist())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "014f4bf8",
   "metadata": {},
   "outputs": [],
   "source": [
    "if (g[\"status\"] == \"ERRO\").any():\n",
    "    raise RuntimeError(\"Gates dos chunks falharam: NÃO publicar.\")\n",
    "\n",
    "cols = [\"chunk_id\", \"fonte\", \"fonte_id\", \"texto\", \"metadata\", \"content_hash\", \"politica_pii\",\n",
    "        \"embedding_status\", \"modelo_embedding\", \"embedding_em\", \"atualizado_em\"]\n",
    "gravar_gold(chunks[cols], \"chunks\")\n",
    "print(chunks[\"embedding_status\"].value_counts().to_string())\n",
    "print(\"removidos desde a última publicação:\", len(removidos))\n",
    "print(\"publicado em s3://warehouse/gold/chunks/\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "02998cbb",
   "metadata": {},
   "source": [
    "### Passo 11 · O contrato com o próximo módulo\n",
    "Esta é a tabela que o banco vetorial vai ter. **Ela é apenas escrita aqui, não é criada.** Repare que `embedding` e o índice ficam em aberto: a **dimensão** depende do modelo de embedding e o **índice** depende do volume; ambos são decisões do próximo módulo."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "7de23e45",
   "metadata": {},
   "outputs": [],
   "source": [
    "DDL_PGVECTOR = \"\"\"\n",
    "CREATE EXTENSION IF NOT EXISTS vector;\n",
    "\n",
    "CREATE TABLE IF NOT EXISTS gold.chunks (\n",
    "    chunk_id          text PRIMARY KEY,         -- 'avaliacao:123' (estável entre execuções)\n",
    "    fonte             text        NOT NULL,     -- de qual tabela veio\n",
    "    fonte_id          text        NOT NULL,\n",
    "    texto             text        NOT NULL,     -- já com PII mascarada\n",
    "    metadata          jsonb       NOT NULL,     -- filtros de busca (produto, nota); sem dado pessoal\n",
    "    content_hash      char(64)    NOT NULL,     -- sha256 do texto normalizado\n",
    "    politica_pii      text        NOT NULL,     -- versão da regra de mascaramento aplicada\n",
    "    embedding         vector(<DIM>),            -- DIM: definido no módulo vetorial (depende do modelo)\n",
    "    modelo_embedding  text,                     -- qual modelo gerou o vetor\n",
    "    embedding_em      timestamptz,\n",
    "    embedding_status  text        NOT NULL DEFAULT 'pendente',\n",
    "    atualizado_em     timestamptz NOT NULL\n",
    ");\n",
    "-- índice (HNSW ou IVFFlat) e filtros por metadata: decididos no módulo vetorial\n",
    "\"\"\"\n",
    "\n",
    "CONTRATO_VETORIAL = {\n",
    "    \"tabela_origem\": \"s3://warehouse/gold/chunks/\",\n",
    "    \"grao\": \"um chunk\",\n",
    "    \"chave\": \"chunk_id\",\n",
    "    \"dono\": \"Dados / IA (exemplo: preencha com o responsável)\",\n",
    "    \"garantias\": [\"texto com PII mascarada pela política \" + VERSAO_POLITICA_PII,\n",
    "                  \"texto com no máximo 1000 caracteres\", \"content_hash = sha256 do texto normalizado\",\n",
    "                  \"metadados sem cliente_id, nome ou e-mail\"],\n",
    "    \"regra_incremental\": \"gerar embedding apenas quando embedding_status = 'pendente'\",\n",
    "    \"fonte_da_verdade\": \"o lake (Parquet); o banco vetorial é derivado e pode ser recriado\",\n",
    "    \"pendente_no_proximo_modulo\": [\"modelo e dimensão do embedding\", \"índice\", \"estratégia de chunking para textos longos\",\n",
    "                                   \"NER para nomes próprios em texto livre\"],\n",
    "    \"ddl_referencia\": DDL_PGVECTOR.strip(),\n",
    "}\n",
    "s3.put_object(Bucket=\"warehouse\", Key=\"gold/_contratos/contrato_vetorial.json\",\n",
    "              Body=json.dumps(CONTRATO_VETORIAL, ensure_ascii=False, indent=2).encode(\"utf-8\"))\n",
    "print(DDL_PGVECTOR)\n",
    "print(\"contrato salvo em s3://warehouse/gold/_contratos/contrato_vetorial.json\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "d86caf66",
   "metadata": {},
   "source": [
    "### Passo 12 · Status: o que o próximo módulo recebe"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "6c807b1e",
   "metadata": {},
   "outputs": [],
   "source": [
    "pub = ler_prefixo(\"warehouse\", \"gold/chunks/\")\n",
    "print(\"chunks publicados :\", len(pub))\n",
    "print(\"estado dos embeddings:\")\n",
    "print(pub[\"embedding_status\"].value_counts().to_string())\n",
    "print(\"\\nobjetos na Gold:\")\n",
    "for chave, tam in listar(\"warehouse\", \"gold/\"):\n",
    "    print(f\"{tam:>9,}  {chave}\")\n",
    "pub[[\"chunk_id\", \"texto\", \"embedding_status\"]].head(3)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "20f07c92",
   "metadata": {},
   "source": [
    "## Qualidade e governança neste notebook\n",
    "- **Contrato antes do código:** grão, dono e descrição de cada coluna. O gate recusa coluna sem descrição.\n",
    "- **Reconciliação:** a receita da Gold foi recalculada por outro caminho e comparada com a Silver.\n",
    "- **Minimização:** e-mail fora da Gold, nome mascarado, `cliente_id` fora dos chunks.\n",
    "- **Menor privilégio:** o Metabase lê com um usuário que só tem `SELECT`.\n",
    "- **Fonte da verdade:** o lake guarda a Gold; o Postgres e o banco vetorial são cópias derivadas.\n",
    "- **Versão da política de PII:** cada chunk registra qual regra o mascarou, então dá para reprocessar quando a regra mudar.\n",
    "\n",
    "## Checkpoint\n",
    "- [ ] `gold/vendas_diarias`, `gold/clientes_360`, `gold/avaliacoes_resumo` e `gold/chunks` existem no bucket `warehouse`.\n",
    "- [ ] Você explica por que a publicação é bloqueada quando um gate falha.\n",
    "- [ ] Você explica por que a PII é mascarada **antes** de gerar embeddings.\n",
    "- [ ] Você explica o que o `content_hash` economiza no próximo módulo.\n",
    "- [ ] Você sabe dizer o que ficou **de fora** de propósito (o banco vetorial) e por quê.\n",
    "\n",
    "➡️ Próximo módulo: embeddings e banco vetorial (pgvector)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}